# 2b단계: v4 통합 → 전체 파이프라인 채점

0단계 정답, 1a ROI, 1b에서 고른 탐지기(yolo26x · imgsz 1920)를 이어 붙여 **v4 파이프라인**을 만들고 채점합니다.

**실행 순서:** 위에서부터 끝까지 순서대로 (중간에 직접 입력할 값 없음)

⚠️ **step2a_tracker_tuning 노트북을 먼저 실행하세요** (tracker_config_v4.json 필요)

**런타임: T4 GPU** · 전체 약 20분 (대부분 ① 탐지 캐시. 한 번 만들면 Drive에 저장돼서 다시 돌릴 때는 건너뜀)

## 공통 설정

In [ ]:
!pip install -q ultralytics
import os, json, glob, shutil, zipfile, re
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment
from google.colab import drive
drive.mount('/content/drive')

# 한글 글꼴 (그래프의 한글이 네모로 깨지지 않게)
import subprocess, matplotlib.font_manager as fm
_font = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(_font):
    subprocess.run("apt-get -qq install -y fonts-nanum > /dev/null", shell=True)
if os.path.exists(_font):
    fm.fontManager.addfont(_font)
    plt.rcParams["font.family"] = "NanumGothic"
else:
    print("⚠️ 한글 글꼴 설치 실패: 그래프의 한글이 네모로 보일 수 있어요")
plt.rcParams["axes.unicode_minus"] = False

In [ ]:
# ===== 경로 (v3 노트북 Cell 2와 동일하게) =====
FOLDER_PATH      = "/content/drive/MyDrive/2025년 홍익 vs 국민 드론뷰"
DRIVE_OUTPUT_DIR = os.path.join(FOLDER_PATH, "pipeline_outputs")   # v3의 tracking_final.json 위치
EVAL_DIR         = os.path.join(DRIVE_OUTPUT_DIR, "eval")           # 이번에 새로 만드는 평가 폴더
IMG_DIR          = os.path.join(EVAL_DIR, "images")                 # 평가용 프레임 20장 (정답 라벨의 기준 이미지)
PRE_DIR          = os.path.join(EVAL_DIR, "prelabels")              # 사전 라벨
for _d in (EVAL_DIR, IMG_DIR, PRE_DIR):
    os.makedirs(_d, exist_ok=True)

# ===== 클래스 =====
# 정답(GT)에는 0~2만 남아야 합니다. 3(Unknown)은 "라벨러가 확인해야 할 박스" 표시용입니다.
CLASS_NAMES = ["Hongik", "Kookmin", "Referee", "Unknown"]
CLS_ID = {n: i for i, n in enumerate(CLASS_NAMES)}

# ===== 평가 클립 =====
# play_range : 득점 플레이 구간 (v3 frame_id 기준, 1부터 시작, 양 끝 포함)
# scorer     : v3에서 득점 선수 경로를 만들 때 쓴 (track_id, 프레임 구간). 득점 선수 '정답 위치'로 사용.
#              ⚠️ 0156의 #171, #183 구간은 v3 노트북 주석에 '추정, 확인 필요'로 되어 있었습니다.
#                 v3의 프레임 브라우저 셀로 확인 후 틀렸다면 여기 숫자를 고치세요.
EVAL_CLIPS = {
    "DJI_20250913133612_0144_D": {
        "clip_key": 34,
        "play_range": (413, 649),
        "scorer": [(2122, (413, 649))],
    },
    "DJI_20250913135020_0156_D": {
        "clip_key": 44,
        "play_range": (167, 525),
        "scorer": [(184, (167, 214)), (171, (215, 313)), (183, (314, 393)), (208, (410, 525))],
    },
}
FRAMES_PER_CLIP = 10

# ===== 매칭 기준 =====
# 선수가 10~25px로 작아서 IoU 0.5는 박스가 조금만 어긋나도 오답이 됩니다.
# 대신 '박스 중심 거리'로 짝을 짓습니다: 거리 <= max(MIN_PX, RATIO × 정답 박스 긴 변)
MATCH_MIN_PX = 10
MATCH_RATIO  = 0.5
SCORER_MATCH_PX = 15   # 득점 선수 정답 위치(발)와 예측 박스 발 위치가 이 거리(px) 이내면 '찾았다'


def video_path_of(stem):
    for ext in (".MP4", ".mp4", ".MOV", ".mov"):
        p = os.path.join(FOLDER_PATH, stem + ext)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"영상을 찾을 수 없습니다: {stem}")

def tracking_json_of(stem, folder=DRIVE_OUTPUT_DIR):
    return os.path.join(folder, f"{stem}_tracking_final.json")

for stem in EVAL_CLIPS:
    print(stem, "| 영상:", os.path.exists(video_path_of(stem)), "| v3 JSON:", os.path.exists(tracking_json_of(stem)))

In [ ]:
# ===== 공통 함수 =====

def load_tracking(json_path):
    """v3 형식 JSON → {frame_id: [player dict, ...]}, field_polygon"""
    with open(json_path, encoding="utf-8") as f:
        data = json.load(f)
    by_frame = {fr["frame_id"]: fr["players"] for fr in data["frames"]}
    poly = np.array(data.get("field_polygon", []), dtype=np.int32)
    return by_frame, poly


def read_frames_sequential(video_path, frame_ids):
    """
    frame_id(1부터 시작)에 해당하는 프레임을 '처음부터 순서대로 읽어서' 가져옵니다.
    cap.set(POS_FRAMES)로 바로 점프하면 드론 영상(H.264/H.265)에서 몇 프레임씩 어긋날 수 있어서,
    v3가 프레임을 센 방식(처음부터 1, 2, 3...)과 똑같이 맞추기 위해 순차로 읽습니다.
    """
    wanted = set(frame_ids)
    last = max(wanted)
    out = {}
    cap = cv2.VideoCapture(video_path)
    fid = 0
    while fid < last:
        ok, frame = cap.read()
        if not ok:
            break
        fid += 1
        if fid in wanted:
            out[fid] = frame
    cap.release()
    missing = wanted - set(out)
    if missing:
        print(f"⚠️ 읽지 못한 프레임: {sorted(missing)}")
    return out


def pick_frames(play_range, n):
    a, b = play_range
    return sorted(set(np.linspace(a, b, n).round().astype(int).tolist()))


def inside(poly, x, y):
    if poly is None or len(poly) < 3:
        return True
    return cv2.pointPolygonTest(poly.reshape(-1, 1, 2), (float(x), float(y)), False) >= 0


def xyxy_to_yolo_line(cls_id, box, W, H):
    x1, y1, x2, y2 = box
    x1, x2 = max(0, x1), min(W, x2)
    y1, y2 = max(0, y1), min(H, y2)
    cx, cy = (x1 + x2) / 2 / W, (y1 + y2) / 2 / H
    w, h = (x2 - x1) / W, (y2 - y1) / H
    return f"{cls_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}"


def yolo_line_to_xyxy(line, W, H):
    c, cx, cy, w, h = line.split()[:5]
    cx, cy, w, h = float(cx) * W, float(cy) * H, float(w) * W, float(h) * H
    return int(c), [cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2]


def center(b):
    return ((b[0] + b[2]) / 2, (b[1] + b[3]) / 2)


def box_iou(a, b):
    """Day 2에서 만든 IoU (xyxy)"""
    iw = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    ih = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = iw * ih
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0


def match_by_center(gt_boxes, pred_boxes):
    """
    정답 박스와 예측 박스를 1:1로 짝짓기 (헝가리안 알고리즘, 중심 거리 기준).
    return: [(gi, pi), ...], 짝 없는 정답 인덱스, 짝 없는 예측 인덱스
    """
    G, P = len(gt_boxes), len(pred_boxes)
    if G == 0 or P == 0:
        return [], list(range(G)), list(range(P))
    gc = np.array([center(b) for b in gt_boxes])
    pc = np.array([center(b) for b in pred_boxes])
    dist = np.linalg.norm(gc[:, None, :] - pc[None, :, :], axis=2)          # (G, P)
    gsize = np.array([max(b[2] - b[0], b[3] - b[1]) for b in gt_boxes])
    thr = np.maximum(MATCH_MIN_PX, MATCH_RATIO * gsize)[:, None]            # 정답 박스별 허용 거리
    cost = np.where(dist <= thr, dist, 1e6)
    gi, pi = linear_sum_assignment(cost)
    pairs = [(g, p) for g, p in zip(gi, pi) if cost[g, p] < 1e6]
    mg = {g for g, _ in pairs}
    mp = {p for _, p in pairs}
    return pairs, [g for g in range(G) if g not in mg], [p for p in range(P) if p not in mp]


print("✅ 공통 함수 준비 완료")

## 평가 준비 (0단계 파트 B와 같은 코드)

In [ ]:
GT_ZIP = os.path.join(EVAL_DIR, "gt_roboflow.zip")
GT_DIR = os.path.join("/content", "gt_extracted")
shutil.rmtree(GT_DIR, ignore_errors=True)
with zipfile.ZipFile(GT_ZIP) as z:
    z.extractall(GT_DIR)


def original_name(fname):
    """Roboflow가 바꾼 파일명 → 원래 이름. 예: 'DJI_..._f00413_jpg.rf.ab12cd.txt' → 'DJI_..._f00413'"""
    base = os.path.basename(fname)
    base = re.split(r"_(?:jpg|jpeg|png)\.rf\.", base, flags=re.IGNORECASE)[0]
    return os.path.splitext(base)[0]


def read_names_yaml(gt_dir):
    """Roboflow data.yaml의 클래스 순서를 읽음 (Roboflow가 순서를 바꿀 수 있어서 이름으로 다시 매핑)"""
    ymls = glob.glob(os.path.join(gt_dir, "**", "data.yaml"), recursive=True)
    if not ymls:
        return CLASS_NAMES
    txt = open(ymls[0], encoding="utf-8").read()
    m = re.search(r"names:\s*\[(.*?)\]", txt, re.S)
    if m:
        return [s.strip().strip("'\"") for s in m.group(1).split(",") if s.strip()]
    return re.findall(r"^\s*-\s*['\"]?([^'\"\n]+?)['\"]?\s*$", txt.split("names:")[1], re.M)


def load_gt(gt_dir):
    """→ {name: {"W","H","boxes":[(class_name, xyxy), ...]}}"""
    rf_names = read_names_yaml(gt_dir)
    gt = {}
    for lbl in glob.glob(os.path.join(gt_dir, "**", "labels", "*.txt"), recursive=True):
        name = original_name(lbl)
        img = cv2.imread(os.path.join(IMG_DIR, name + ".jpg"))   # 원본 해상도 기준으로 복원
        if img is None:
            print("⚠️ 원본 이미지를 찾을 수 없음:", name); continue
        H, W = img.shape[:2]
        boxes = []
        for line in open(lbl).read().strip().splitlines():
            if not line.strip():
                continue
            c, b = yolo_line_to_xyxy(line, W, H)
            boxes.append((rf_names[c], b))
        gt[name] = {"W": W, "H": H, "boxes": boxes}
    return gt


GT = load_gt(GT_DIR)
_cnt = Counter(c for v in GT.values() for c, _ in v["boxes"])
print(f"정답 이미지 {len(GT)}장, 박스 {sum(_cnt.values())}개 → {dict(_cnt)}")
if _cnt.get("Unknown"):
    print(f"⚠️ 정답에 Unknown이 {_cnt['Unknown']}개 남아 있습니다. 탐지 평가에는 포함되지만 팀 정확도에서는 제외됩니다.")

In [ ]:
EVAL_TEAM = ["Hongik", "Kookmin", "Referee"]


def detection_and_team_metrics(run_json_by_stem):
    """정답 vs 예측: 탐지(클래스 무관) P/R/F1 + 짝지어진 박스의 클래스 정확도 + 혼동행렬"""
    tp = fp = fn = 0
    conf_mat = Counter()                     # (정답 클래스, 예측 클래스)
    per_frame = []
    for stem, jpath in run_json_by_stem.items():
        by_frame, _ = load_tracking(jpath)
        for name, g in GT.items():
            if not name.startswith(stem):
                continue
            fid = int(name.split("_f")[-1])
            preds = by_frame.get(fid, [])
            gb = [b for _, b in g["boxes"]]
            pb = [p["bbox"] for p in preds]
            pairs, ug, up = match_by_center(gb, pb)
            tp += len(pairs); fn += len(ug); fp += len(up)
            for gi, pi in pairs:
                conf_mat[(g["boxes"][gi][0], preds[pi]["type"])] += 1
            per_frame.append({"image": name, "gt": len(gb), "pred": len(pb), "TP": len(pairs), "FN": len(ug), "FP": len(up)})
    P = tp / (tp + fp) if tp + fp else 0.0
    R = tp / (tp + fn) if tp + fn else 0.0
    F1 = 2 * P * R / (P + R) if P + R else 0.0
    team_pairs = {k: v for k, v in conf_mat.items() if k[0] in EVAL_TEAM}
    correct = sum(v for (g, p), v in team_pairs.items() if g == p)
    team_acc = correct / sum(team_pairs.values()) if team_pairs else 0.0
    ref_gt = sum(1 for v in GT.values() for c, _ in v["boxes"] if c == "Referee")
    ref_hit = conf_mat.get(("Referee", "Referee"), 0)
    return {
        "det_precision": round(P, 3), "det_recall": round(R, 3), "det_f1": round(F1, 3),
        "TP": tp, "FP": fp, "FN": fn,
        "team_acc": round(team_acc, 3),
        "referee_recall": round(ref_hit / ref_gt, 3) if ref_gt else None,
    }, conf_mat, pd.DataFrame(per_frame)


def scorer_track_metrics(stem, run_json, cfg, verbose=True):
    """
    득점 선수 추적 품질 (라벨링 없이 측정).
    - 정답 위치: v3에서 사람이 확인해 이어붙인 득점 선수의 발 위치 (cfg["scorer"])
    - 평가할 결과(run_json)에서 매 프레임 그 위치에 가장 가까운 박스의 track_id를 읽음

    지표
    - scorer_ids / scorer_id_switches : '정답 위치가 확실한 프레임'에서만 계산 (믿을 수 있는 값)
    - scorer_coverage : v3가 놓친 구간은 앞뒤 위치를 직선으로 이어 정답 위치를 '추정'해서 포함
                        → 그 구간에서는 근처의 다른 선수가 잡힐 수 있으니 참고용(근사값)
    """
    ref_by_frame, _ = load_tracking(tracking_json_of(stem))       # 정답 경로는 항상 v3 기준
    run_by_frame, _ = load_tracking(run_json)
    known = {}
    for tid, (a, b) in cfg["scorer"]:
        for fid in range(a, b + 1):
            for p in ref_by_frame.get(fid, []):
                if p["track_id"] == tid:
                    known[fid] = p["feet"]
    if not known:
        print(f"⚠️ {stem}: v3 JSON에서 득점 선수 track_id를 찾지 못했습니다. EVAL_CLIPS의 scorer 값을 확인하세요.")
        return {"scorer_frames": 0, "scorer_coverage": 0.0, "scorer_ids": 0, "scorer_id_switches": 0}
    span = range(min(known), max(known) + 1)
    kf = np.array(sorted(known))
    kx = np.array([known[f][0] for f in kf]); ky = np.array([known[f][1] for f in kf])

    seq = []   # (frame, 찾은 track_id 또는 None, 정답 위치가 확실한지)
    for fid in span:
        gx, gy = (known[fid] if fid in known else (np.interp(fid, kf, kx), np.interp(fid, kf, ky)))
        best, bd = None, 1e9
        for p in run_by_frame.get(fid, []):
            d = np.hypot(p["feet"][0] - gx, p["feet"][1] - gy)
            if d < bd:
                best, bd = p, d
        tid = best["track_id"] if best is not None and bd <= SCORER_MATCH_PX else None
        seq.append((fid, tid, fid in known))

    sure = [t for _, t, k in seq if k and t is not None]
    switches = sum(1 for a, b in zip(sure, sure[1:]) if a != b)
    covered = sum(1 for _, t, _ in seq if t is not None)

    if verbose:   # 어느 ID가 어느 구간을 맡았는지 (v3 노트북의 SEGMENTS와 같은 형태) → 영상으로 확인할 때 사용
        runs, cur = [], None
        for fid, t, _ in seq:
            if cur and cur[0] == t and cur[2] == fid - 1:
                cur[2] = fid
            else:
                cur = [t, fid, fid]; runs.append(cur)
        desc = ", ".join(f"{'없음' if t is None else '#' + str(t)}({a}~{b})" for t, a, b in runs)
        print(f"  [{stem}] 득점 선수 경로: {desc}")

    return {
        "scorer_frames": len(seq),
        "scorer_coverage": round(covered / len(seq), 3),
        "scorer_ids": len(set(sure)),
        "scorer_id_switches": switches,
    }


def segment_stats(stem, run_json, cfg):
    """플레이 구간 안의 고유 ID 수, Unknown 비율 (라벨링 없이 측정)"""
    by_frame, _ = load_tracking(run_json)
    a, b = cfg["play_range"]
    ids, types = set(), Counter()
    for fid in range(a, b + 1):
        for p in by_frame.get(fid, []):
            ids.add(p["track_id"]); types[p["type"]] += 1
    total = sum(types.values())
    return {"segment_unique_ids": len(ids),
            "segment_unknown_ratio": round(types["Unknown"] / total, 3) if total else 0.0}


RESULTS_CSV = os.path.join(EVAL_DIR, "eval_results.csv")


def evaluate_run(run_name, run_json_by_stem, note=""):
    """
    한 번의 실험 결과를 채점해서 eval_results.csv에 한 줄 추가합니다.
    run_json_by_stem: {stem: 그 실험이 만든 tracking_final.json 경로}
    """
    det, conf_mat, per_frame = detection_and_team_metrics(run_json_by_stem)
    row = {"run": run_name, "note": note, **det}
    for stem, jpath in run_json_by_stem.items():
        cfg = EVAL_CLIPS[stem]
        k = cfg["clip_key"]
        for key, val in {**scorer_track_metrics(stem, jpath, cfg), **segment_stats(stem, jpath, cfg)}.items():
            row[f"{key}_{k}"] = val
    df_new = pd.DataFrame([row])
    if os.path.exists(RESULTS_CSV):
        old = pd.read_csv(RESULTS_CSV)
        old = old[old["run"] != run_name]                      # 같은 이름으로 다시 돌리면 덮어쓰기
        df_new = pd.concat([old, df_new], ignore_index=True)
    df_new.to_csv(RESULTS_CSV, index=False, encoding="utf-8-sig")
    return row, conf_mat, per_frame


def show_confusion(conf_mat):
    rows = EVAL_TEAM + (["Unknown"] if any(g == "Unknown" for g, _ in conf_mat) else [])
    cols = CLASS_NAMES
    return pd.DataFrame([[conf_mat.get((r, c), 0) for c in cols] for r in rows],
                        index=[f"정답:{r}" for r in rows], columns=[f"예측:{c}" for c in cols])

print("✅ 평가 함수 준비 완료")

In [ ]:
# 틀린 곳을 눈으로 확인: 초록=맞게 찾음(TP), 빨강=놓친 정답(FN), 파랑=잘못 잡음(FP)
def show_errors(name, run_json_by_stem):
    stem = name.split("_f")[0]
    fid = int(name.split("_f")[-1])
    by_frame, poly = load_tracking(run_json_by_stem[stem])
    preds = by_frame.get(fid, [])
    g = GT[name]
    pairs, ug, up = match_by_center([b for _, b in g["boxes"]], [p["bbox"] for p in preds])
    img = cv2.imread(os.path.join(IMG_DIR, name + ".jpg"))
    cv2.polylines(img, [poly.reshape(-1, 1, 2)], True, (0, 255, 0), 1)
    for gi, pi in pairs:
        b = preds[pi]["bbox"]; cv2.rectangle(img, (int(b[0]), int(b[1])), (int(b[2]), int(b[3])), (0, 200, 0), 2)
    for gi in ug:
        b = g["boxes"][gi][1]; cv2.rectangle(img, (int(b[0]), int(b[1])), (int(b[2]), int(b[3])), (0, 0, 255), 2)
    for pi in up:
        b = preds[pi]["bbox"]; cv2.rectangle(img, (int(b[0]), int(b[1])), (int(b[2]), int(b[3])), (255, 0, 0), 2)
    plt.figure(figsize=(16, 9)); plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(f"{name}  TP={len(pairs)}  FN={len(ug)}  FP={len(up)}"); plt.axis("off"); plt.show()

## ByteTrack
v3는 Ultralytics 안의 ByteTrack을 썼어요. 여기서는 저장해둔 탐지 결과에 설정만 바꿔 여러 번 돌리려고,
같은 규칙을 numpy로 옮긴 버전을 씁니다. 셀 끝의 자체 점검이 통과하는지 확인하세요.

In [ ]:
# ===== ByteTrack (논문 · Ultralytics 구현과 같은 규칙을 그대로 옮긴 버전) =====
# 저장해둔 탐지 결과에 바로 돌릴 수 있도록 numpy만으로 작성했습니다.
#   1) 칼만 필터로 기존 트랙의 이번 위치 예측
#   2) 높은 점수 박스와 IoU 짝짓기 (헝가리안)
#   3) 짝 못 찾은 트랙은 낮은 점수 박스와 한 번 더 짝짓기  ← ByteTrack의 핵심
#   4) 남은 높은 점수 박스 중 new_track_thresh 이상만 새 트랙으로 등록
#   5) track_buffer 프레임 동안 못 찾은 트랙은 삭제

class KalmanXYAH:
    """상태: [cx, cy, 종횡비 a, 높이 h, 각 속도 4개]. 등속 모델."""
    def __init__(self):
        self.F = np.eye(8)
        for i in range(4):
            self.F[i, 4 + i] = 1.0
        self.H = np.eye(4, 8)
        self.wp, self.wv = 1.0 / 20, 1.0 / 160

    def initiate(self, z):
        h = z[3]
        mean = np.r_[z, np.zeros(4)]
        std = [2 * self.wp * h, 2 * self.wp * h, 1e-2, 2 * self.wp * h,
               10 * self.wv * h, 10 * self.wv * h, 1e-5, 10 * self.wv * h]
        return mean, np.diag(np.square(std))

    def predict(self, mean, cov):
        h = mean[3]
        q = [self.wp * h, self.wp * h, 1e-2, self.wp * h, self.wv * h, self.wv * h, 1e-5, self.wv * h]
        return self.F @ mean, self.F @ cov @ self.F.T + np.diag(np.square(q))

    def update(self, mean, cov, z):
        h = mean[3]
        R = np.diag(np.square([self.wp * h, self.wp * h, 1e-1, self.wp * h]))
        S = self.H @ cov @ self.H.T + R
        K = np.linalg.solve(S, (cov @ self.H.T).T).T
        mean = mean + K @ (z - self.H @ mean)
        cov = cov - K @ S @ K.T
        return mean, cov


def xyxy_to_xyah(b):
    w, h = b[2] - b[0], b[3] - b[1]
    return np.array([(b[0] + b[2]) / 2, (b[1] + b[3]) / 2, w / max(h, 1e-6), h], float)


def xyah_to_xyxy(m):
    cx, cy, a, h = m[:4]
    w = a * h
    return np.array([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], float)


def iou_matrix_np(a, b):
    a = np.asarray(a, float).reshape(-1, 4); b = np.asarray(b, float).reshape(-1, 4)
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    lt = np.maximum(a[:, None, :2], b[None, :, :2]); rb = np.minimum(a[:, None, 2:], b[None, :, 2:])
    wh = np.clip(rb - lt, 0, None); inter = wh[..., 0] * wh[..., 1]
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1]); area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (area_a[:, None] + area_b[None, :] - inter + 1e-9)


def linear_assign(cost, thresh):
    if cost.size == 0:
        return [], list(range(cost.shape[0])), list(range(cost.shape[1]))
    c = np.where(cost <= thresh, cost, 1e6)
    r, k = linear_sum_assignment(c)
    m = [(i, j) for i, j in zip(r, k) if c[i, j] <= thresh]
    mi = {i for i, _ in m}; mj = {j for _, j in m}
    return m, [i for i in range(cost.shape[0]) if i not in mi], [j for j in range(cost.shape[1]) if j not in mj]


class Track:
    _next_id = 0
    TRACKED, LOST, REMOVED = 0, 1, 2

    def __init__(self, box, score, det_idx, kf):
        self.kf, self.box, self.score, self.det_idx = kf, np.asarray(box, float), score, det_idx
        self.mean = self.cov = None
        self.state, self.activated, self.track_id = Track.TRACKED, False, -1
        self.start_frame = self.end_frame = 0

    def predicted_box(self):
        return xyah_to_xyxy(self.mean) if self.mean is not None else self.box

    def predict(self):
        m = self.mean.copy()
        if self.state != Track.TRACKED:
            m[7] = 0.0                          # 놓친 트랙은 높이 변화 속도 0
        self.mean, self.cov = self.kf.predict(m, self.cov)

    def activate(self, frame_id):
        Track._next_id += 1
        self.track_id = Track._next_id
        self.mean, self.cov = self.kf.initiate(xyxy_to_xyah(self.box))
        self.state = Track.TRACKED
        self.activated = frame_id == 1
        self.start_frame = self.end_frame = frame_id

    def update(self, det, frame_id):
        self.mean, self.cov = self.kf.update(self.mean, self.cov, xyxy_to_xyah(det.box))
        self.box, self.score, self.det_idx = det.box, det.score, det.det_idx
        self.state, self.activated, self.end_frame = Track.TRACKED, True, frame_id


class ByteTracker:
    def __init__(self, track_high_thresh=0.5, track_low_thresh=0.1, new_track_thresh=0.6,
                 track_buffer=30, match_thresh=0.8, fuse_score=True, frame_rate=30):
        self.hi, self.lo, self.new = track_high_thresh, track_low_thresh, new_track_thresh
        self.max_lost = int(frame_rate / 30.0 * track_buffer)
        self.match, self.fuse = match_thresh, fuse_score
        self.kf = KalmanXYAH()
        self.tracked, self.lost = [], []
        self.frame_id = 0
        Track._next_id = 0

    def _dist(self, tracks, dets, fuse):
        d = 1 - iou_matrix_np([t.predicted_box() for t in tracks], [x.box for x in dets])
        if fuse and len(dets):
            d = 1 - (1 - d) * np.array([x.score for x in dets])[None, :]
        return d

    def update(self, boxes, scores):
        """boxes (N,4) xyxy, scores (N,) → [(track_id, det_idx), ...] 이번 프레임에 확정된 트랙"""
        self.frame_id += 1
        f = self.frame_id
        boxes = np.asarray(boxes, float).reshape(-1, 4); scores = np.asarray(scores, float).reshape(-1)
        high = [Track(boxes[i], scores[i], i, self.kf) for i in np.where(scores >= self.hi)[0]]
        low = [Track(boxes[i], scores[i], i, self.kf) for i in np.where((scores > self.lo) & (scores < self.hi))[0]]

        unconfirmed = [t for t in self.tracked if not t.activated]
        confirmed = [t for t in self.tracked if t.activated]
        pool = confirmed + [t for t in self.lost if t not in confirmed]
        for t in pool:
            t.predict()

        refound, activated_now = [], []
        # 1차: 높은 점수 박스
        m, u_t, u_d = linear_assign(self._dist(pool, high, self.fuse), self.match)
        for i, j in m:
            t = pool[i]
            if t.state != Track.TRACKED:
                refound.append(t)
            t.update(high[j], f); activated_now.append(t)
        # 2차: 남은 '추적 중' 트랙 × 낮은 점수 박스
        r_tracked = [pool[i] for i in u_t if pool[i].state == Track.TRACKED]
        m2, u_t2, _ = linear_assign(self._dist(r_tracked, low, False), 0.5)
        for i, j in m2:
            r_tracked[i].update(low[j], f); activated_now.append(r_tracked[i])
        for i in u_t2:
            r_tracked[i].state = Track.LOST
        # 확정 안 된(1프레임짜리) 트랙 × 남은 높은 점수 박스
        rest_high = [high[j] for j in u_d]
        m3, u_un, u_d3 = linear_assign(self._dist(unconfirmed, rest_high, self.fuse), 0.7)
        for i, j in m3:
            unconfirmed[i].update(rest_high[j], f); activated_now.append(unconfirmed[i])
        for i in u_un:
            unconfirmed[i].state = Track.REMOVED
        # 새 트랙
        new_tracks = []
        for j in u_d3:
            d = rest_high[j]
            if d.score >= self.new:
                d.activate(f); new_tracks.append(d)
        # 오래 못 찾은 트랙 삭제
        for t in pool:
            if t.state == Track.LOST and f - t.end_frame > self.max_lost:
                t.state = Track.REMOVED

        everyone = {id(t): t for t in self.tracked + self.lost + new_tracks}
        self.tracked = [t for t in everyone.values() if t.state == Track.TRACKED]
        self.lost = [t for t in everyone.values() if t.state == Track.LOST]
        return [(t.track_id, t.det_idx) for t in self.tracked if t.activated and t.end_frame == f]


def _bytetrack_selftest():
    """작은 합성 장면으로 동작 확인: 점수가 잠깐 낮아지거나 몇 프레임 사라져도 ID 유지, 약한 새 박스는 새 ID 안 받음"""
    tr = ByteTracker(track_high_thresh=0.5, track_low_thresh=0.1, new_track_thresh=0.6, track_buffer=30)
    ids = defaultdict(set)
    for f in range(1, 61):
        boxes, scores = [], []
        x = 100 + 3 * f                                     # 선수 A: 오른쪽으로 이동
        if not (30 <= f < 36):                              # 30~35프레임 동안 사라짐(가려짐)
            boxes.append([x, 100, x + 12, 130]); scores.append(0.3 if 15 <= f < 20 else 0.8)   # 15~19 낮은 점수
        boxes.append([300, 200 + f, 312, 230 + f]); scores.append(0.9)                      # 선수 B
        if f == 40:
            boxes.append([500, 50, 512, 80]); scores.append(0.4)                            # 약한 오탐 1번
        for tid, di in tr.update(boxes, scores):
            ids["A" if boxes[di][0] < 290 else ("B" if boxes[di][0] < 400 else "X")].add(tid)
    assert len(ids["A"]) == 1 and len(ids["B"]) == 1 and "X" not in ids, dict(ids)
    print("✅ ByteTrack 자체 점검 통과:", {k: sorted(v) for k, v in ids.items()})

_bytetrack_selftest()

---
# 2단계 + v4: 탐지 캐시 → 추적기 비교 → 팀 분류 → 전체 채점

| 순서 | 내용 | 시간 (T4) |
|---|---|---|
| ① | yolo26x(1920)로 두 클립 **전체** 탐지 + 팀 분류 특징 저장 (한 번만) | 약 15분 |
| ② | **2a 노트북에서 튜닝 클립으로 고른 추적기 설정**과 v3 설정을 각각 적용 | 1분 |
| ③ | v3 RF 팀 분류기 + 트랙 다수결 + 인원 규칙 → v4 JSON | 1분 |
| ④ | `evaluate_run`으로 채점, v3와 비교 | 1분 |

v4 = 1a ROI + yolo26x imgsz 1920 + 2a에서 튜닝한 ByteTrack + v3 팀 분류 방식

⚠️ **먼저 `step2a_tracker_tuning` 노트북을 실행해서 `tracker_config_v4.json`을 만들어두세요.**

In [ ]:
import time, pickle, joblib
from ultralytics import YOLO

DET_WEIGHTS = "yolo26x.pt"
DET_IMGSZ = 1920
DET_CONF_CACHE = 0.05          # 낮게 저장해두고 추적기 설정에서 걸러 씀 (low_thresh 비교용)
FULL_CLIP = True               # True: v3처럼 클립 처음부터 끝까지 처리 / False: 플레이 구간 + 앞 WARMUP_FRAMES만 (빠른 확인용)
WARMUP_FRAMES = 90
CACHE_DIR = os.path.join(EVAL_DIR, "cache")
os.makedirs(CACHE_DIR, exist_ok=True)

ROI_V4 = {}
for stem in EVAL_CLIPS:
    p = os.path.join(EVAL_DIR, "runs", "step1a_roi", f"{stem}_tracking_final.json")
    assert os.path.exists(p), f"1a 결과가 없습니다: {p}"
    _, ROI_V4[stem] = load_tracking(p)

TEAM_CLF_PATH = os.path.join(DRIVE_OUTPUT_DIR, "team_classifier.joblib")
assert os.path.exists(TEAM_CLF_PATH), f"v3 팀 분류기가 없습니다: {TEAM_CLF_PATH}"
team_classifier = joblib.load(TEAM_CLF_PATH)
print("v3 팀 분류기 클래스:", list(team_classifier.classes_))

In [ ]:
# ===== v3 노트북 Cell 6의 팀 분류 특징 함수 (그대로 옮김) =====
LOWER_GREEN = np.array([25, 25, 25])
UPPER_GREEN = np.array([95, 255, 255])
REFEREE_HAT_RATIO_THRESH = 0.30


def get_torso_head_crops(player_crop):
    h, w, _ = player_crop.shape
    head_crop = player_crop[0:int(h * 0.25), :]
    torso_crop = player_crop[int(h * 0.20):int(h * 0.75), int(w * 0.10):int(w * 0.90)]
    if torso_crop.size == 0:
        torso_crop = player_crop
    return head_crop, torso_crop


def cluster_balance_ratio(player_crop, k=2):
    _, torso = get_torso_head_crops(player_crop)
    if torso.size == 0:
        return 0.0
    hsv_torso = cv2.cvtColor(torso, cv2.COLOR_BGR2HSV)
    keep = cv2.inRange(hsv_torso, LOWER_GREEN, UPPER_GREEN) == 0
    bgr = torso[keep]
    if len(bgr) < 20:
        bgr = torso.reshape(-1, 3)
    if len(bgr) < k:
        return 0.0
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 10, 1.0)
    _, labels, _ = cv2.kmeans(bgr.astype(np.float32), k, None, criteria, 3, cv2.KMEANS_PP_CENTERS)
    counts = np.bincount(labels.flatten(), minlength=k)
    return float(counts.min() / counts.sum())


def is_referee(player_crop):
    head_crop, _ = get_torso_head_crops(player_crop)
    if head_crop.size == 0:
        return False, 0.0
    head_hsv = cv2.cvtColor(head_crop, cv2.COLOR_BGR2HSV)
    white = cv2.inRange(head_hsv, np.array([0, 0, 160]), np.array([180, 60, 255]))
    ratio = np.sum(white > 0) / (head_crop.shape[0] * head_crop.shape[1] + 1e-5)
    if ratio >= REFEREE_HAT_RATIO_THRESH:
        return True, ratio
    if ratio >= REFEREE_HAT_RATIO_THRESH * 0.5 and cluster_balance_ratio(player_crop) >= 0.35:
        return True, ratio
    return False, ratio


def extract_features(player_crop):
    _, torso = get_torso_head_crops(player_crop)
    if torso.size == 0:
        return None
    hsv = cv2.cvtColor(torso, cv2.COLOR_BGR2HSV)
    keep = cv2.inRange(hsv, LOWER_GREEN, UPPER_GREEN) == 0
    pixels = hsv[keep]
    if len(pixels) < 20:
        pixels = hsv.reshape(-1, 3)
    if len(pixels) == 0:
        return None
    h_hist, _ = np.histogram(pixels[:, 0], bins=8, range=(0, 180), density=True)
    s_hist, _ = np.histogram(pixels[:, 1], bins=8, range=(0, 256), density=True)
    v_hist, _ = np.histogram(pixels[:, 2], bins=8, range=(0, 256), density=True)
    stats = [pixels[:, 0].mean(), pixels[:, 0].std(), pixels[:, 1].mean(), pixels[:, 1].std(),
             pixels[:, 2].mean(), pixels[:, 2].std()]
    return np.concatenate([h_hist, s_hist, v_hist, stats, [is_referee(player_crop)[1], cluster_balance_ratio(player_crop)]])

print("✅ 팀 분류 특징 함수 준비 완료")

In [ ]:
# ① 탐지 캐시: 프레임마다 (박스, 점수, 팀 분류 확률) 저장. 이미 있으면 건너뜀.
def cache_path(stem):
    tag = "full" if FULL_CLIP else f"warm{WARMUP_FRAMES}"
    return os.path.join(CACHE_DIR, f"{stem}_y26x{DET_IMGSZ}_conf{DET_CONF_CACHE}_{tag}.pkl")


def build_cache(stem, cfg, model):
    a, b = cfg["play_range"]
    start, end = (1, 10 ** 9) if FULL_CLIP else (max(1, a - WARMUP_FRAMES), b)
    local = os.path.join("/content/_scratch_video", os.path.basename(video_path_of(stem)))
    os.makedirs(os.path.dirname(local), exist_ok=True)
    if not os.path.exists(local):
        shutil.copy(video_path_of(stem), local)            # Drive에서 바로 읽으면 느려서 로컬로 복사
    cap = cv2.VideoCapture(local)
    frames, fid, t0 = {}, 0, time.time()
    classes = list(team_classifier.classes_)
    while fid < end:
        ok, img = cap.read()
        if not ok:
            break
        fid += 1                                           # v3와 같은 방식으로 1부터 셈 (순차 읽기)
        if fid < start:
            continue
        r = model.predict(img, imgsz=DET_IMGSZ, conf=DET_CONF_CACHE, classes=[0],
                          max_det=300, verbose=False)[0]
        bx = r.boxes.xyxy.cpu().numpy().reshape(-1, 4); sc = r.boxes.conf.cpu().numpy().reshape(-1)
        keep = [i for i, bb in enumerate(bx) if inside(ROI_V4[stem], (bb[0] + bb[2]) / 2, bb[3])]
        bx, sc = bx[keep], sc[keep]
        probs = np.full((len(bx), len(classes)), np.nan)
        for i, bb in enumerate(bx):
            x1, y1, x2, y2 = [int(v) for v in bb]
            crop = img[max(0, y1):max(0, y2), max(0, x1):max(0, x2)]
            if crop.size == 0:
                continue
            feat = extract_features(crop)
            if feat is not None and np.all(np.isfinite(feat)):
                probs[i] = team_classifier.predict_proba([feat])[0]
        frames[fid] = {"boxes": bx, "scores": sc, "probs": probs}
        if (fid - start) % 200 == 0:
            print(f"  [{stem}] {fid}프레임  ({time.time() - t0:.0f}s)")
    cap.release()
    end = fid
    data = {"stem": stem, "start": start, "end": end, "classes": classes, "frames": frames,
            "weights": DET_WEIGHTS, "imgsz": DET_IMGSZ}
    with open(cache_path(stem), "wb") as f:
        pickle.dump(data, f)
    print(f"✅ {stem}: {len(frames)}프레임 캐시 저장 ({time.time() - t0:.0f}s)")
    return data


CACHE = {}
_model = None
for stem, cfg in EVAL_CLIPS.items():
    if os.path.exists(cache_path(stem)):
        with open(cache_path(stem), "rb") as f:
            CACHE[stem] = pickle.load(f)
        print(f"⏭️ {stem}: 저장된 캐시 사용 ({len(CACHE[stem]['frames'])}프레임)")
        continue
    if _model is None:
        _model = YOLO(DET_WEIGHTS)
    CACHE[stem] = build_cache(stem, cfg, _model)

In [ ]:
# ② 추적기: 2a에서 튜닝 클립(0143, 0157)으로 고른 설정을 불러와 적용 (평가 클립에서는 설정을 고르지 않음)
TRACKER_CFG_PATH = os.path.join(EVAL_DIR, "tracker_config_v4.json")
assert os.path.exists(TRACKER_CFG_PATH), "tracker_config_v4.json이 없습니다. step2a_tracker_tuning 노트북을 먼저 실행하세요."
_tcfg = json.load(open(TRACKER_CFG_PATH, encoding="utf-8"))
TRACKER_CONFIGS = {"v3설정": _tcfg["v3_params"], "튜닝": _tcfg["params"]}
CHOSEN = "튜닝"
print("튜닝 클립:", _tcfg["tune_clips"])
print("선택 규칙:", _tcfg["rule"])
print("v3 설정 :", _tcfg["v3_params"])
print("튜닝 설정:", _tcfg["params"], f"({_tcfg['chosen']})")


def run_tracker(stem, params):
    """캐시된 탐지에 추적기를 돌림 → {frame_id: [(track_id, det_idx), ...]}"""
    c = CACHE[stem]
    tr = ByteTracker(**params, fuse_score=True, frame_rate=30)
    out = {}
    for fid in range(c["start"], c["end"] + 1):
        fr = c["frames"].get(fid, {"boxes": np.zeros((0, 4)), "scores": np.zeros(0)})
        out[fid] = tr.update(fr["boxes"], fr["scores"])
    return out


TRACK_RESULTS = {name: {stem: run_tracker(stem, p) for stem in EVAL_CLIPS} for name, p in TRACKER_CONFIGS.items()}
print("✅ 추적 완료")

In [ ]:
# ③ 팀 분류: 박스별 RF 확률 → 트랙별 다수결 → 인원 규칙 (v3 Cell 12와 같은 흐름)
MAX_REFEREE_PER_FRAME = 3
MAX_PLAYERS_PER_TEAM_PER_FRAME = 11


def build_v4_json(stem, tracks, run_dir):
    c = CACHE[stem]; classes = c["classes"]
    # 1) 박스별 1차 라벨
    per = {}
    for fid, lst in tracks.items():
        fr = c["frames"].get(fid)
        for tid, di in lst:
            pr = fr["probs"][di]
            lab = classes[int(np.nanargmax(pr))] if np.all(np.isfinite(pr)) else "Unknown"
            per[(fid, tid)] = (lab, pr, fr["boxes"][di], float(fr["scores"][di]))
    # 2) 트랙별 다수결
    votes = defaultdict(Counter)
    for (fid, tid), (lab, *_) in per.items():
        votes[tid][lab] += 1
    majority = {tid: v.most_common(1)[0][0] for tid, v in votes.items()}
    # 3) 인원 규칙
    frames_out, demoted = [], 0
    for fid in sorted(tracks):
        items = []
        for tid, di in tracks[fid]:
            lab0, pr, bb, sc = per[(fid, tid)]
            items.append({"tid": tid, "label": majority[tid], "pr": pr, "bb": bb, "sc": sc})

        def prob(it, lab):
            return float(it["pr"][classes.index(lab)]) if lab in classes and np.all(np.isfinite(it["pr"])) else 0.0

        refs = sorted([it for it in items if it["label"] == "Referee"], key=lambda it: -prob(it, "Referee"))
        for it in refs[MAX_REFEREE_PER_FRAME:]:          # 심판 초과분 → 심판을 뺀 클래스 중 확률이 높은 쪽
            alt = [l for l in classes if l != "Referee"]
            it["label"] = max(alt, key=lambda l: prob(it, l)) if alt else "Unknown"; demoted += 1
        for team in ("Hongik", "Kookmin"):
            mem = sorted([it for it in items if it["label"] == team], key=lambda it: -prob(it, team))
            for it in mem[MAX_PLAYERS_PER_TEAM_PER_FRAME:]:
                it["label"] = "Unknown"; demoted += 1
        players = []
        for it in items:
            x1, y1, x2, y2 = [int(round(v)) for v in it["bb"]]
            players.append({"track_id": int(it["tid"]), "type": it["label"], "bbox": [x1, y1, x2, y2],
                            "feet": [(x1 + x2) // 2, y2], "conf": it["sc"]})
        frames_out.append({"frame_id": int(fid), "players": players})
    data = {"video_name": os.path.basename(video_path_of(stem)), "fps": 30,
            "field_polygon": np.asarray(ROI_V4[stem]).tolist(), "frames": frames_out,
            "notes": [f"v4: {DET_WEIGHTS} imgsz{DET_IMGSZ} + ByteTrack({run_dir.split('/')[-1]}) + v3 RF 분류 + 인원 규칙",
                      f"프레임 {c['start']}~{c['end']} ({'클립 전체' if FULL_CLIP else f'플레이 구간 + 앞 {WARMUP_FRAMES}프레임'})"]}
    os.makedirs(run_dir, exist_ok=True)
    out = os.path.join(run_dir, f"{stem}_tracking_final.json")
    with open(out, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False)
    print(f"  {stem}: 인원 규칙으로 조정 {demoted}건 → {out}")
    return out


RUNS_V4 = {}
for name, run_name in [("v3설정", "v4_tracker_v3cfg"), ("튜닝", "v4")]:
    run_dir = os.path.join(EVAL_DIR, "runs", run_name)
    RUNS_V4[run_name] = {stem: build_v4_json(stem, TRACK_RESULTS[name][stem], run_dir) for stem in EVAL_CLIPS}

In [ ]:
# ④ 채점 + 전체 비교
row_a, conf_a, pf_a = evaluate_run("v4_tracker_v3cfg", RUNS_V4["v4_tracker_v3cfg"],
                                   note=f"1a ROI + {DET_WEIGHTS} {DET_IMGSZ} + ByteTrack(v3 설정) + v3 RF 분류")
row_v4, conf_v4, pf_v4 = evaluate_run("v4", RUNS_V4["v4"],
                                      note=f"1a ROI + {DET_WEIGHTS} {DET_IMGSZ} + ByteTrack(튜닝: {_tcfg['chosen']}) + v3 RF 분류")

COLS = ["run", "det_precision", "det_recall", "det_f1", "TP", "FP", "FN", "team_acc", "referee_recall",
        "scorer_ids_34", "scorer_id_switches_34", "scorer_coverage_34",
        "scorer_ids_44", "scorer_id_switches_44", "scorer_coverage_44",
        "segment_unique_ids_34", "segment_unique_ids_44", "segment_unknown_ratio_34", "segment_unknown_ratio_44"]
res = pd.read_csv(RESULTS_CSV)
order = [r for r in ["v3_baseline", "step1a_roi", "v4_tracker_v3cfg", "v4"] if r in set(res["run"])]
display(res.set_index("run").loc[order, [c for c in COLS if c != "run"]].T)

print("\n=== v4 팀 분류 혼동행렬 ===")
display(show_confusion(conf_v4))

In [ ]:
# 같은 프레임에서 오류 확인: v4가 가장 많이 놓친 2장
for name in pf_v4.sort_values("FN", ascending=False)["image"].head(2):
    show_errors(name, RUNS_V4["v4"])

## 결과 공유
비교표(`v3_baseline` → `step1a_roi` → `v4_tracker_v3cfg` → `v4`), 혼동행렬, 오류 이미지를 보여주세요.

**읽는 법**
- `step1a_roi` → `v4_tracker_v3cfg`: **탐지기 교체 효과** (yolov8x 960 → yolo26x 1920, 추적기 설정은 v3와 같음)
- `v4_tracker_v3cfg` → `v4`: **추적기 조정 효과**
- 득점 선수 지표(`scorer_*`)와 고유 ID 수: 추적 품질
- `team_acc`, `referee_recall`: 분류기는 v3 그대로라 크게 안 바뀌는 게 정상 (3단계 과제)